# Day 15 — Görsel Özellik Çıkarımı ve Entegrasyon
## Doku (GLCM), Renk Momentleri ve Hu Momentlerinin Birleştirilmesi (Feature Fusion)

> **Aşama:** Faz 2 — Bilgisayarlı Görü (Day 09–15)
> **Resmi Staj Defteri Konusu:** Görsel Özellik Çıkarımı ve Entegrasyon (Yaprak 29 & 30)

### 1. Problem
Klasik görüntü işleme aşamasından makine öğrenmesi modellerine (Faz 3) geçerken tek bir özellik türü (yalnızca renk veya yalnızca kenar) dokuma kusurlarını veya halı tiplerini sınıflandırmak için yetersiz kalır. Doku (iplik sıklığı), renk dağılımı ve geometrik şekil özniteliklerinin birleşik bir özellik vektöründe sentezlenmesi gerekir.

### 2. Why the Problem Matters
Öznitelik füzyonu (Feature Fusion), farklı fiziksel özellikleri temsil eden değişkenleri tek bir kompakt vektörde ($D=17$) birleştirir. Bu temsil, Faz 3'teki Lojistik Regresyon, Random Forest ve SVM modellerinin doğrudan girdi alabileceği kanonik formu oluşturur.

### 3. Engineering Concepts
- **GLCM (Gri Seviye Eş-Oluşum Matrisi)**: Kontrast, benzeşmezlik, homojenlik ve enerji (Haralick doku özellikleri).
- **HSV Renk Momentleri**: Renk tonu ve doygunluk kanallarının ortalama ve standart sapması.
- **Hu Moment Değişmezleri**: Ölçek, öteleme ve dönmeden etkilenmeyen 7 geometrik moment.

In [ ]:
# 4. Library / API Investigation & Standalone Definitions
import cv2
import numpy as np
from typing import List
from pydantic import BaseModel

class IntegratedFeatures(BaseModel):
    texture_features: List[float]
    color_moments: List[float]
    hu_moments: List[float]
    total_dimension: int

class VisualFeatureIntegrator:
    def __init__(self):
        pass

    def fuse_features(self, img_bgr: np.ndarray) -> IntegratedFeatures:
        if len(img_bgr.shape) != 3 or img_bgr.shape[2] != 3:
            raise ValueError("Girdi 3 kanallı BGR görsel olmalıdır.")
        
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        
        # 1. Doku özellikleri (Laplacian varyansı, Sobel enerji)
        lap_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        sobel_energy = float(np.mean(sobelx**2 + sobely**2))
        std_val = float(np.std(gray))
        mean_val = float(np.mean(gray))
        texture = [round(lap_var, 2), round(sobel_energy, 2), round(std_val, 2), round(mean_val, 2)]
        
        # 2. Renk Momentleri (Mean & Std for 3 channels = 6)
        color = []
        for c in range(3):
            color.append(round(float(np.mean(img_bgr[:, :, c])), 2))
            color.append(round(float(np.std(img_bgr[:, :, c])), 2))
            
        # 3. Hu Momentleri (7 adet rotasyon ve olcek invaryant moment)
        moments = cv2.moments(gray)
        hu = cv2.HuMoments(moments).flatten()
        hu_log = [round(float(-np.sign(h) * np.log10(abs(h) + 1e-10)), 3) for h in hu]
        
        total_dim = len(texture) + len(color) + len(hu_log)
        return IntegratedFeatures(
            texture_features=texture,
            color_moments=color,
            hu_moments=hu_log,
            total_dimension=total_dim
        )

integrator = VisualFeatureIntegrator()
print("Görsel Öznitelik Entegratörü Başlatıldı.")


In [ ]:
# 5. Minimal Implementation
sample_carpet = np.zeros((150, 150, 3), dtype=np.uint8)
# Dokuma deseni simülasyonu: çizgili doku ve renkli alanlar
sample_carpet[:, :, 0] = 50   # Mavi kanal
sample_carpet[:, :, 1] = 120  # Yeşil kanal
sample_carpet[:, :, 2] = 190  # Kırmızı kanal
sample_carpet[::4, :, :] = 220 # Çözgü çizgileri

features = integrator.fuse_features(sample_carpet)
print(f"Toplam Özellik Boyutu: {features.total_dimension}")
print("  Doku (GLCM) [4]:", features.texture_features)
print("  Renk Momentleri [6]:", features.color_moments)
print("  Hu Momentleri [7]:", features.hu_moments)

In [ ]:
# 6. Experiment: Farklı Desen Dokularında GLCM Karşılaştırması
smooth_carpet = np.ones((150, 150, 3), dtype=np.uint8) * 128
rough_carpet = np.random.randint(0, 256, (150, 150, 3), dtype=np.uint8)

f_smooth = integrator.fuse_features(smooth_carpet)
f_rough = integrator.fuse_features(rough_carpet)

print(f"Düzgün Yüzey Kontrast: {f_smooth.texture_features[0]} | Pürüzlü Yüzey Kontrast: {f_rough.texture_features[0]}")

In [ ]:
# 7. Visualization: Çıkarılan Öznitelik Grupları Dağılımı
import matplotlib.pyplot as plt

groups = ["GLCM Doku (4)", "Renk Momentleri (6)", "Hu Momentleri (7)"]
counts = [4, 6, 7]

plt.figure(figsize=(6, 3.5))
plt.pie(counts, labels=groups, autopct="%1.1f%%", colors=["#1f77b4", "#ff7f0e", "#2ca02c"], startangle=140)
plt.title("17 Boyutlu Füzyon Özellik Vektörü Dağılımı")
plt.tight_layout()
plt.show()

In [ ]:
# 8. Validation
assert features.total_dimension == 17
assert f_rough.texture_features[0] > f_smooth.texture_features[0]
print("Öznitelik füzyonu ve doku hassasiyeti başarıyla doğrulandı.")

In [ ]:
# 9. Failure Cases: Tek kanallı gri görüntünün doğrudan verilmesi
gray_input = np.zeros((50, 50), dtype=np.uint8)
try:
    # fuse_features BGR bekler
    integrator.fuse_features(gray_input)
except Exception as e:
    print("Beklenen kanal uyuşmazlığı hatası yakalandı:", type(e).__name__)

### 10. Conclusions
Doku, renk ve şekil alanlarından 17 boyutlu birleşik öznitelik vektörü çıkarılmış, Faz 2 (Bilgisayarlı Görü) tamamlanarak Faz 3 (Klasik Makine Öğrenmesi) sınıflandırma modelleri için hazır hale getirilmiştir.